# Week 6 Lab 02: Accuracy vs F1 on Imbalanced Data

**Scenario:** Cordwell Home and Hardware's support queue tags a small fraction of tickets as **high risk**: a completed installation may be causing safety or property damage right now. Most tickets are routine. Your job is to build a classifier that catches the high-risk ones, and, more importantly, to learn why the most popular metric in machine learning is the wrong tool for judging it.

**Estimated duration:** 120 minutes.

By the end of this lab you will be able to:

- Assemble an imbalanced dataset for a binary classification problem and inspect its class balance.
- Build a majority-class baseline with `DummyClassifier` and explain what it proves.
- Compute and interpret accuracy, precision, recall, and F1 for competing models on the same test set.
- Explain, with numbers from your own run, why accuracy can be misleading when the positive class is rare.
- Use `class_weight="balanced"` to trade false alarms for caught cases, and defend that trade in business terms.

**How this connects to Lab 01.** Previously you learned what the four metrics mean and how a decision threshold moves them. In this lab, the class balance changes from 50-50 to 90-10, and the question changes from "what do the metrics mean" to "which metric should drive the decision." Three deliberate differences from the first lab: the split is two-way instead of three-way (nothing gets threshold-tuned against a validation set today), the baseline uses sklearn's `DummyClassifier` instead of your hand-rolled version (same idea, production idiom), and the imbalance is present from the first cell instead of arriving at the end.

## How this notebook works

Cells marked **PROVIDED** are plumbing: run them and move on. Cells marked **TASK** contain a function contract and a `raise NotImplementedError`; replace the raise with your implementation. Each task is followed by an **apply** cell that uses your function and a **checks** cell that grades it.

The check harness never crashes the notebook. Unwritten tasks print `[TODO]`, wrong answers print `[FAIL]` with the reason, and your running score appears wherever `summary()` is called. A fresh Run All on this notebook completes with zero errors and 3 of 26 checks passing; your goal is 26 of 26.

Two hint files sit next to this notebook. `HINTS.md` offers three escalating nudges per task. `HINTS_DETAILED.md` shows the working core of each task with commentary. Pick one tier per task; reading both wastes time.

## Part 0: Environment

Everything runs locally on CPU. No Docker services, no local LLM server, no network calls, and nothing here touches a GPU. The MLflow tracking server arrives later this week; today stays dependency-light on purpose.

If the imports cell fails, run `pip install -r requirements.txt` in your environment and restart the kernel.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# One seed drives corpus generation, the split, and the models, so your
# numbers match the worked target output exactly. As in Lab 1.1, the seed
# was chosen so the models make an instructive mix of mistakes.
RANDOM_SEED = 7

print(f"scikit-learn {sklearn.__version__}")
print(f"pandas       {pd.__version__}")
print(f"numpy        {np.__version__}")

In [ ]:
CHECK_RESULTS = {}

def check(name: str, fn) -> None:
    """Run a zero-argument callable and record a named PASS or FAIL.

    Unimplemented tasks (NotImplementedError) and missing upstream results
    (NameError, or None placeholders being poked) report as TODO, not FAIL.
    """
    try:
        ok = bool(fn())
    except NotImplementedError:
        CHECK_RESULTS[name] = False
        print(f"[TODO] {name}: task not implemented yet")
        return
    except Exception as exc:
        CHECK_RESULTS[name] = False
        if isinstance(exc, NameError) or "NoneType" in str(exc):
            print(f"[TODO] {name}: depends on an earlier task")
        else:
            print(f"[FAIL] {name}: {type(exc).__name__}: {exc}")
        return
    CHECK_RESULTS[name] = ok
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")

def attempt(label: str, fn):
    """Run a student function for an apply cell; return its result or None."""
    try:
        return fn()
    except NotImplementedError:
        print(f"[TODO] {label}: implement the task above, then re-run this cell.")
        return None
    except Exception as exc:
        print(f"[ERROR] {label}: {type(exc).__name__}: {exc}")
        return None

def summary() -> None:
    total = len(CHECK_RESULTS)
    passed = sum(CHECK_RESULTS.values())
    print(f"Checks passing: {passed}/{total}")

print("Check harness ready.")

In [ ]:
check(
    "env: scikit-learn 1.6 or newer",
    lambda: tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 6),
)
check(
    "env: pandas 2.0 or newer",
    lambda: tuple(int(p) for p in pd.__version__.split(".")[:2]) >= (2, 0),
)
check(
    "env: numpy 2.0 or newer",
    lambda: tuple(int(p) for p in np.__version__.split(".")[:2]) >= (2, 0),
)
summary()

## Part 1: The ticket corpus

Cordwell agents write a short case note for every support contact. A downstream router should tag the small fraction that describe **active safety or property risk** (label 1, high risk) so a supervisor queue sees them within minutes. Everything else is routine (label 0), including tickets that are unhappy, complaint-shaped, or full of installation vocabulary but not dangerous.

That last clause is what makes this corpus honest. The negative class includes **hard negatives**: real complaints (a misaligned cabinet door, a disputed invoice, a crew that parked badly) that sound negative but put nobody at risk. The positive class includes **hard positives**: soft-spoken reports (a faint gas smell, a warm outlet cover, a chirping smoke detector someone disconnected) where the customer themselves is not alarmed but the situation is. A model that just learns "complaint words mean high risk" or "calm words mean routine" will get these wrong, exactly like a naive router would in production.

The sentence pools and ticket builders below are **PROVIDED**. Read the pools; the hard cases are worth thirty seconds of your attention. Assembling them into a corpus is your first task.

In [ ]:
OPENINGS_NEUTRAL = [
    "The customer contacted the Cordwell support line about a recent purchase.",
    "The customer reached out through the Cordwell help center chat.",
    "A call came in to the Cordwell customer care queue this morning.",
    "The customer submitted a support request through the Cordwell mobile app.",
    "The customer visited the service desk and the associate opened a ticket.",
    "An email from the customer was converted into a support ticket.",
]

OPENINGS_CONCERNED = [
    "The customer called the Cordwell support line sounding worried.",
    "The customer contacted Cordwell support and asked to speak with someone right away.",
    "A call came in from a customer who said the matter felt urgent to them.",
    "The customer opened a ticket and marked it as needing prompt attention.",
]

DEPARTMENTS = [
    "flooring", "kitchen and bath", "lumber", "paint", "lighting",
    "hardware", "outdoor and garden", "appliances", "tools", "windows and doors",
]

PRODUCTS = [
    "laminate planks", "vinyl tile", "a cabinet set", "a quartz countertop",
    "a ceiling fan", "a smart thermostat", "a circular saw", "deck boards",
    "a storm door", "a bathroom vanity", "a water heater", "a pressure washer",
    "a garage door opener", "a gas range", "a shower enclosure",
]

# Clear negatives: routine questions with no problem content at all.
ISSUES_ROUTINE = [
    "The customer has a question about store hours and the holiday schedule.",
    "The customer wants to confirm whether an item is in stock before driving over.",
    "The customer is asking if price matching applies to a competitor's advertisement.",
    "The customer needs help understanding the difference between two product lines.",
    "The customer is checking whether installation services cover their zip code.",
    "The customer wants to reschedule a delivery window for a small order.",
    "The customer is trying to apply a promotion code at checkout and needs guidance.",
    "The customer is asking for advice on stain colors to match existing trim.",
    "The customer wants to add an extended protection plan to an earlier order.",
    "The customer is asking how to enroll in the loyalty program online.",
]

# Hard negatives: genuine complaints and installation vocabulary, zero danger.
ISSUES_HARD_NEGATIVE = [
    "The customer is unhappy that trim pieces from a recent installation show visible seams.",
    "The customer says the crew left sawdust and packaging behind after the installation visit.",
    "The customer is disputing a labor charge that appeared on the installation invoice.",
    "The customer asks what the warranty would cover if the new unit ever failed down the road.",
    "The customer wants to know how to report a problem if the installation develops issues later.",
    "The customer mentions the cabinet doors sit slightly misaligned and wants an adjustment visit.",
    "The customer says the installation ran two hours late and wants to give feedback about scheduling.",
    "The customer reports a small cosmetic scratch on the appliance door that was there at delivery.",
    "The customer says the paint color looks different on the wall than the sample and wants options.",
    "The customer complains that the installation crew parked in the neighbor's driveway.",
]

# Clear positives: unmistakable active safety or property damage.
ISSUES_CLEAR_POSITIVE = [
    "The customer reports that the recently installed dishwasher is leaking and damaging the kitchen floor.",
    "The customer says a ceiling fan installed last week is wobbling badly and making grinding noises.",
    "The customer reports a burning smell coming from behind the newly installed electrical panel cover.",
    "The customer says loose deck boards from a recent installation caused a guest to trip and fall.",
    "The customer reports serious water damage after a shower installation was not properly sealed.",
    "The customer says the new garage door has been closing on its own without warning.",
    "The customer reports that the gas range hookup smells strongly of gas and the family has headaches.",
    "The customer says newly anchored stair handrails pull away from the wall when used.",
]

# Hard positives: hedged, soft-spoken reports of real risk.
ISSUES_HARD_POSITIVE = [
    "The customer mentions a faint gas smell near the new range but is not sure it is anything.",
    "The customer says an outlet cover near the new microwave feels warm to the touch sometimes.",
    "The customer notices the new shelving unit leans away from the wall when the kids climb near it.",
    "The customer reports occasional dripping inside the wall cavity after the shower installation, mostly when it rains.",
    "The customer says the smoke detector was disconnected during the ceiling fan installation and still chirps.",
    "The customer mentions the new deck railing flexes more than expected when someone leans on it.",
    "The customer says the water heater makes a popping sound and the floor around it feels damp.",
    "The customer reports small sparks from the disposal switch, though it still seems to work fine.",
]

# Shared by both classes on purpose: process language carries no label signal.
PROCESS = [
    "The associate pulls up the order history and reviews the installation record.",
    "The agent confirms the customer's contact details and preferred callback window.",
    "Notes from the original service visit are attached to the ticket for reference.",
    "The associate reviews the account and checks for any related open tickets.",
    "The agent summarizes the conversation in the ticket notes for the next reviewer.",
    "The customer's order number and store location are verified in the system.",
]

RESOLUTIONS = [
    "The agent documents the details and provides the customer with a reference number.",
    "A follow-up contact is scheduled according to standard procedure.",
    "The ticket is routed onward and the customer is told what to expect next.",
    "The associate thanks the customer and confirms the summary before closing the call.",
    "The customer is sent a written recap of the conversation by email.",
]

def pick_kind(rng: random.Random, label: int) -> str:
    """Roll a difficulty kind for one ticket: 40 percent of positives and
    25 percent of negatives are hard cases; the rest are clear."""
    threshold = 0.4 if label == 1 else 0.25
    return "hard" if rng.random() < threshold else "clear"

def make_ticket(rng: random.Random, label: int, kind: str) -> str:
    """Assemble one agent-written case note for the given label and kind."""
    dept = rng.choice(DEPARTMENTS)
    product = rng.choice(PRODUCTS)
    if label == 1:
        opening = rng.choice(OPENINGS_CONCERNED if kind == "clear" else OPENINGS_NEUTRAL)
        issue = rng.choice(ISSUES_CLEAR_POSITIVE if kind == "clear" else ISSUES_HARD_POSITIVE)
    else:
        opening = rng.choice(OPENINGS_NEUTRAL if kind == "clear" else (OPENINGS_NEUTRAL + OPENINGS_CONCERNED))
        issue = rng.choice(ISSUES_ROUTINE if kind == "clear" else ISSUES_HARD_NEGATIVE)
    context = f"The ticket concerns {product} from the {dept} department."
    body = [context, issue] + rng.sample(PROCESS, 2)
    rng.shuffle(body)
    return " ".join([opening] + body + [rng.choice(RESOLUTIONS)])

_preview = random.Random(0)
print("Example clear positive (high risk):\n ", make_ticket(_preview, 1, "clear"))
print()
print("Example hard positive (high risk):\n ", make_ticket(_preview, 1, "hard"))
print()
print("Example hard negative (routine):\n ", make_ticket(_preview, 0, "hard"))

### Task 1: Assemble the corpus

Build the dataset the rest of the lab stands on: 500 tickets, 10 percent high risk. Note what the hardness plumbing already did for you: `pick_kind` decides how difficult each ticket is and `make_ticket` writes it. Your job is the dataset engineering around them: exact positive count, generation, shuffle, DataFrame.

Why the shuffle matters: you will generate all the positives first, and an unshuffled corpus would put every positive in the first 50 rows. Downstream code that peeks at `head()` would see a wildly unrepresentative sample, and any tooling that splits data by position instead of at random would produce garbage silently.

In [ ]:
def build_corpus(
    n_docs: int = 500,
    positive_fraction: float = 0.10,
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """Generate an imbalanced Cordwell ticket corpus.

    Steps:
      1. Create one random.Random(seed) instance and use it for everything.
      2. Compute n_pos = round(n_docs * positive_fraction).
      3. For each index i in range(n_docs): the label is 1 while i < n_pos,
         else 0. Roll kind = pick_kind(rng, label), write the ticket with
         make_ticket(rng, label, kind), and collect a dict with keys
         "text", "label", "kind".
      4. Shuffle the collected rows in place with rng.shuffle.
      5. Return pd.DataFrame(rows).

    Returns:
      DataFrame with columns ["text", "label", "kind"], length n_docs,
      containing exactly n_pos rows with label 1, in shuffled order.
    """
    rng = random.Random(seed)
    n_pos = round(n_docs * positive_fraction)
    rows = []
    for i in range(n_docs):
        label = 1 if i < n_pos else 0
        kind = pick_kind(rng, label)
        rows.append({"text": make_ticket(rng, label, kind), "label": label, "kind": kind})
    rng.shuffle(rows)
    return pd.DataFrame(rows)

In [ ]:
tickets_df = attempt("build_corpus", lambda: build_corpus())

if tickets_df is not None:
    print(f"Corpus size: {len(tickets_df)}")
    print()
    print("Class distribution (percent):")
    dist = (tickets_df["label"].map({0: "routine", 1: "high_risk"})
            .value_counts(normalize=True) * 100).round(1)
    print(dist.to_string())
    print()
    print("Difficulty mix within each class (counts):")
    print(tickets_df.groupby("label")["kind"].value_counts().to_string())
    print()
    print("First two tickets:")
    for text in tickets_df["text"].head(2):
        print(" -", text[:160], "...")

In [ ]:
# PROVIDED: class balance bar chart. The point of the picture is the gap.
if tickets_df is not None:
    counts = tickets_df["label"].map({0: "routine", 1: "high_risk"}).value_counts()
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.bar(counts.index, counts.values, color=["#4878a8", "#c44e52"])
    ax.set_ylabel("ticket count")
    ax.set_title("Cordwell support tickets by class")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 6, str(v), ha="center")
    plt.tight_layout()
    plt.show()

In [ ]:
check(
    "task 1: corpus has 500 rows and the columns text, label, kind",
    lambda: tickets_df.shape == (500, 3) and list(tickets_df.columns) == ["text", "label", "kind"],
)
check(
    "task 1: exactly 50 high-risk tickets (10 percent of 500)",
    lambda: int(tickets_df["label"].sum()) == 50,
)
check(
    "task 1: rows are shuffled, not grouped by label",
    lambda: int(tickets_df["label"].head(50).sum()) < 50,
)
check(
    "task 1: same seed reproduces the same corpus",
    lambda: build_corpus().equals(build_corpus()),
)
summary()

**Quick questions before moving on.** About what fraction of tickets are high risk? If a classifier always predicted routine, what accuracy would it score on this data? Hold your answer; you will build exactly that classifier in Task 3 and grade your prediction.

## Worked target output

Everything is seeded, so a correct implementation reproduces these numbers exactly. Code toward this target; if a check fails, compare your output here first.

**Corpus (Task 1):** 500 tickets, 450 routine (90.0 percent), 50 high risk (10.0 percent). Difficulty mix by class: routine splits 321 clear and 129 hard, high risk splits 29 clear and 21 hard.

**Split (Task 2):** 400 train and 100 test, stratified: 40 of 400 train tickets are high risk (10.0 percent) and 10 of 100 test tickets are high risk (10.0 percent).

**Majority baseline (Tasks 3 and 4):** predicts routine for everything.

```
Confusion matrix (rows true, columns predicted):
[[90  0]
 [10  0]]
accuracy   0.900
precision  0.000
recall     0.000
f1         0.000
```

**Logistic regression, default settings (Task 5):**

```
[[90  0]
 [ 8  2]]
accuracy   0.920
precision  1.000
recall     0.200
f1         0.333
```

**Logistic regression with class_weight="balanced" (Task 6):**

```
[[89  1]
 [ 0 10]]
accuracy   0.990
precision  0.909
recall     1.000
f1         0.952
```

**Comparison table (Task 7):**

```
                   model  accuracy  precision  recall    f1
       majority baseline      0.90      0.000     0.0 0.000
     logistic regression      0.92      1.000     0.2 0.333
LR class_weight=balanced      0.99      0.909     1.0 0.952
```

## Part 2: Train and test split

### Task 2: Stratified 80-20 split

Two-way this time, not the morning's three-way. The reason is worth internalizing: a validation set exists to absorb tuning decisions, and the main path of this lab tunes nothing, it compares three fixed recipes. The moment you start tuning (stretch goal 3 does), the tuning has to happen inside the training data.

Stratification matters more at 90-10 than it did at 50-50 this morning. An unstratified 100-row test sample from a 10 percent population could easily carry 5 positives or 15, and every recall number downstream would inherit that coin flip.

In [ ]:
def split_tickets(df: pd.DataFrame, seed: int = RANDOM_SEED):
    """Split the corpus into train and test sets, stratified by label.

    Steps:
      1. Extract X from the "text" column and y from the "label" column,
         both as numpy arrays via .to_numpy().
      2. One train_test_split call: test_size=0.20, stratify on the labels,
         random_state=seed.

    Returns:
      (X_train, X_test, y_train, y_test), sized 400, 100, 400, 100.
    """
    X = df["text"].to_numpy()
    y = df["label"].to_numpy()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=seed
    )
    return X_train, X_test, y_train, y_test

In [ ]:
split_result = attempt("split_tickets", lambda: split_tickets(tickets_df))

if split_result is not None:
    X_train, X_test, y_train, y_test = split_result
    print(f"Train: n={len(X_train)}, high risk={int(y_train.sum())} ({100 * y_train.mean():.1f}%)")
    print(f"Test : n={len(X_test)}, high risk={int(y_test.sum())} ({100 * y_test.mean():.1f}%)")
else:
    X_train = X_test = y_train = y_test = None

In [ ]:
check(
    "task 2: sizes are 400 train and 100 test",
    lambda: len(X_train) == 400 and len(X_test) == 100,
)
check(
    "task 2: stratified, 40 and 10 high-risk tickets land in train and test",
    lambda: int(y_train.sum()) == 40 and int(y_test.sum()) == 10,
)
check(
    "task 2: same seed reproduces the same split",
    lambda: np.array_equal(split_tickets(tickets_df)[3], y_test),
)
summary()

## Part 3: The majority-class baseline

### Task 3: Fit a DummyClassifier

This morning you built the always-predict-the-majority baseline by hand with `np.bincount` and `np.full_like`, so you know there is no magic in it. sklearn ships the same idea as `DummyClassifier`, and the production version has one real advantage: it walks and talks like every other estimator, so it drops into any pipeline, harness, or tracking system unchanged. From here on, the dummy is how you will baseline every classifier you build in this course.

`strategy="most_frequent"` reproduces exactly your hand-rolled version. The estimator ignores the input features entirely; it fits on labels alone. Pass `random_state=seed` anyway: this strategy never uses it, but some strategies do (one of the stretch goals meets such a strategy), and passing it uniformly costs nothing.

In [ ]:
def fit_majority_baseline(X_train, y_train, seed: int = RANDOM_SEED) -> DummyClassifier:
    """Fit a majority-class baseline.

    Steps:
      1. Construct DummyClassifier(strategy="most_frequent", random_state=seed).
      2. Fit it on X_train, y_train.

    Returns:
      The fitted DummyClassifier.
    """
    baseline = DummyClassifier(strategy="most_frequent", random_state=seed)
    baseline.fit(X_train, y_train)
    return baseline

In [ ]:
majority_clf = attempt("fit_majority_baseline", lambda: fit_majority_baseline(X_train, y_train))

if majority_clf is not None:
    y_pred_dummy = majority_clf.predict(X_test)
    print(f"Distinct predicted labels: {sorted(set(y_pred_dummy.tolist()))}")
    print(f"Predicted high-risk tickets: {int(y_pred_dummy.sum())} of {len(y_pred_dummy)}")
else:
    y_pred_dummy = None

In [ ]:
check(
    "task 3: returns a fitted DummyClassifier",
    lambda: majority_clf.classes_ is not None and isinstance(majority_clf, DummyClassifier),
)
check(
    "task 3: strategy is most_frequent",
    lambda: majority_clf.get_params()["strategy"] == "most_frequent",
)
check(
    "task 3: predicts routine (0) for every test ticket",
    lambda: len(y_pred_dummy) == 100 and int(y_pred_dummy.sum()) == 0,
)
summary()

## Part 4: One evaluation function for every model

### Task 4: evaluate_predictions

This morning the metrics function and the confusion matrix were two tasks. This afternoon they merge into one function that will be called three times, once per model, so that every model in the comparison is graded by literally the same code. When an evaluation function is shared, a difference between two rows of the comparison table can only come from the models. That is the property that makes the table trustworthy, and it is the property MLflow runs will need later this week.

Same conventions as the morning: round to 3 decimals inside the function, and pass `zero_division=0` to precision, recall, and F1. The dummy model is about to make the zero-division case real: it never predicts positive, so precision's denominator is zero, and without that argument this cell would drown in warnings.

In [ ]:
def evaluate_predictions(y_true, y_pred):
    """Score one model's predictions.

    Steps:
      1. Build a dict with keys "accuracy", "precision", "recall", "f1".
         Each value is the matching sklearn score rounded to 3 decimals.
         precision, recall, and f1 take zero_division=0.
      2. Build the confusion matrix with confusion_matrix(y_true, y_pred),
         truth first.

    Returns:
      (metrics_dict, cm) where cm is the 2x2 numpy array.
    """
    metrics = {
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 3),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 3),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 3),
    }
    cm = confusion_matrix(y_true, y_pred)
    return metrics, cm

In [ ]:
dummy_eval = attempt("evaluate_predictions", lambda: evaluate_predictions(y_test, y_pred_dummy))

if dummy_eval is not None:
    metrics_dummy, cm_dummy = dummy_eval
    print("Majority baseline on the test set")
    print("Confusion matrix (rows true, columns predicted):")
    print(cm_dummy)
    for k, v in metrics_dummy.items():
        print(f"{k:<10} {v:.3f}")

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_dummy, display_labels=["routine", "high_risk"]
    )
    fig, ax = plt.subplots(figsize=(5.0, 4.2))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title("Majority baseline")
    plt.tight_layout()
    plt.show()
else:
    metrics_dummy, cm_dummy = None, None

In [ ]:
_fx_true = np.array([1, 0, 0, 0, 1, 1])
_fx_pred = np.array([1, 0, 0, 1, 0, 1])

check(
    "task 4: metrics correct on a hand-checkable fixture",
    lambda: evaluate_predictions(_fx_true, _fx_pred)[0]
    == {"accuracy": 0.667, "precision": 0.667, "recall": 0.667, "f1": 0.667},
)
check(
    "task 4: confusion matrix correct on the fixture",
    lambda: np.array_equal(evaluate_predictions(_fx_true, _fx_pred)[1], np.array([[2, 1], [1, 2]])),
)
check(
    "task 4: dummy scores 0.9 accuracy with precision, recall, and f1 all 0",
    lambda: dict(metrics_dummy) == {"accuracy": 0.9, "precision": 0.0, "recall": 0.0, "f1": 0.0},
)
check(
    "task 4: dummy confusion matrix is [[90, 0], [10, 0]]",
    lambda: cm_dummy.shape == (2, 2) and np.array_equal(cm_dummy, np.array([[90, 0], [10, 0]])),
)
summary()

**Stop and register what just happened.** A model implementable as `return 0` scored 90 percent accuracy while catching zero of the ten dangerous situations in the test set. Every one of those ten customers has a leaking pipe, a warm outlet, or a gas smell, and every one of them was told, in effect, that everything is fine. If your dashboard shows one number and that number is accuracy, this model looks like a strong start. This is why the majority baseline opens every honest evaluation: it is the score that zero intelligence buys on your data, and 90 percent of it comes free with the imbalance.

## Part 5: A real model

### Task 5: TF-IDF and logistic regression pipeline

Word-for-word the pipeline you built this morning: two named steps, `"tfidf"` with `ngram_range=(1, 2)` and `"clf"` with `max_iter=1000` and the seed. Building it a second time from a blank cell is the point; this pattern should start leaving your fingers without reference. The interesting part of this task is not the code, it is what the metrics do to it on 90-10 data.

In [ ]:
def build_and_fit_pipeline(X_train, y_train, seed: int = RANDOM_SEED) -> Pipeline:
    """Build and fit the TF-IDF plus logistic regression pipeline.

    Steps:
      1. Pipeline with two named steps:
         "tfidf": TfidfVectorizer(ngram_range=(1, 2))
         "clf":   LogisticRegression(max_iter=1000, random_state=seed)
      2. Fit the whole pipeline on X_train, y_train.

    Returns:
      The fitted Pipeline.
    """
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000, random_state=seed)),
    ])
    pipeline.fit(X_train, y_train)
    return pipeline

In [ ]:
lr_pipeline = attempt("build_and_fit_pipeline", lambda: build_and_fit_pipeline(X_train, y_train))

if lr_pipeline is not None:
    y_pred_lr = lr_pipeline.predict(X_test)
    lr_eval = attempt("evaluate_predictions", lambda: evaluate_predictions(y_test, y_pred_lr))
    if lr_eval is not None:
        metrics_lr, cm_lr = lr_eval
        print("Logistic regression on the test set")
        print("Confusion matrix (rows true, columns predicted):")
        print(cm_lr)
        for k, v in metrics_lr.items():
            print(f"{k:<10} {v:.3f}")

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm_lr, display_labels=["routine", "high_risk"]
        )
        fig, ax = plt.subplots(figsize=(5.0, 4.2))
        disp.plot(ax=ax, colorbar=False)
        ax.set_title("Logistic regression, default settings")
        plt.tight_layout()
        plt.show()

        print()
        print(classification_report(
            y_test, y_pred_lr, target_names=["routine", "high_risk"], zero_division=0
        ))
    else:
        metrics_lr, cm_lr = None, None
else:
    y_pred_lr, metrics_lr, cm_lr = None, None, None

In [ ]:
check(
    "task 5: pipeline has steps named tfidf and clf with the required settings",
    lambda: lr_pipeline.named_steps["tfidf"].ngram_range == (1, 2)
    and lr_pipeline.named_steps["clf"].max_iter == 1000
    and isinstance(lr_pipeline, Pipeline),
)
check(
    "task 5: pipeline is fitted and predicts one label per test ticket",
    lambda: len(y_pred_lr) == 100,
)
check(
    "task 5: flags exactly 2 test tickets as high risk",
    lambda: int(y_pred_lr.sum()) == 2,
)
summary()

**Read your own table before scrolling on.** Accuracy moved from 0.900 to 0.920, a gain of two points, and the model looks slightly better than the dummy. Recall moved from 0.000 to 0.200: it caught 2 of the 10 dangerous situations and missed 8, including, if you scan the misses, most of the soft-spoken hard positives. Precision is a perfect 1.000, and it is a decoy: the model raised two alarms all test set and both happened to be real. Trustworthy silence is still silence. A support manager reading only the accuracy column would approve this model; a support manager reading the recall column would ask who is calling those 8 customers back.

Why so conservative? The training objective minimized overall loss on data where 90 percent of the answers are routine. Being wrong about a rare positive barely moves the loss, so the cheapest fit is one that hedges toward routine. The model did exactly what it was asked; the next task changes what it is asked.

## Part 6: Rebalancing the objective

### Task 6: class_weight="balanced"

This was a stretch goal this morning; on 90-10 data it is the main event. One argument, `class_weight="balanced"` on the LogisticRegression, rescales each class's contribution to the training loss inversely to its frequency. At 10 percent positives, each high-risk ticket now weighs about 9 times as much as a routine one, and ignoring the rare class stops being the cheap way to minimize loss. Everything else stays identical to Task 5, which is exactly what makes the before-and-after comparison clean.

In [ ]:
def build_and_fit_balanced_pipeline(X_train, y_train, seed: int = RANDOM_SEED) -> Pipeline:
    """Task 5's pipeline with one change: class_weight="balanced" on the classifier.

    Returns:
      The fitted Pipeline.
    """
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000, random_state=seed, class_weight="balanced")),
    ])
    pipeline.fit(X_train, y_train)
    return pipeline

In [ ]:
bal_pipeline = attempt(
    "build_and_fit_balanced_pipeline",
    lambda: build_and_fit_balanced_pipeline(X_train, y_train),
)

if bal_pipeline is not None:
    y_pred_bal = bal_pipeline.predict(X_test)
    bal_eval = attempt("evaluate_predictions", lambda: evaluate_predictions(y_test, y_pred_bal))
    if bal_eval is not None:
        metrics_bal, cm_bal = bal_eval
        print("Balanced logistic regression on the test set")
        print("Confusion matrix (rows true, columns predicted):")
        print(cm_bal)
        for k, v in metrics_bal.items():
            print(f"{k:<10} {v:.3f}")

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm_bal, display_labels=["routine", "high_risk"]
        )
        fig, ax = plt.subplots(figsize=(5.0, 4.2))
        disp.plot(ax=ax, colorbar=False)
        ax.set_title("Logistic regression, class_weight balanced")
        plt.tight_layout()
        plt.show()
    else:
        metrics_bal, cm_bal = None, None
else:
    y_pred_bal, metrics_bal, cm_bal = None, None, None

In [ ]:
check(
    "task 6: classifier uses class_weight balanced",
    lambda: bal_pipeline.named_steps["clf"].class_weight == "balanced",
)
check(
    "task 6: flags 11 test tickets as high risk (10 real, 1 false alarm)",
    lambda: int(y_pred_bal.sum()) == 11,
)
check(
    "task 6: recall on the high-risk class is 1.0",
    lambda: metrics_bal["recall"] == 1.0,
)
summary()

## Part 7: The comparison table

### Task 7: build_comparison_table

Three models, one test set, one evaluation function; the last step is putting the rows side by side, because side by side is where the argument lives. This function takes a list of (name, metrics_dict) pairs so it works for three models today, five models tomorrow, and whatever an MLflow experiment holds later this week.

In [ ]:
def build_comparison_table(results) -> pd.DataFrame:
    """Assemble model results into one comparison table.

    Args:
      results: list of (model_name, metrics_dict) pairs, where each
        metrics_dict has the keys accuracy, precision, recall, f1.

    Steps:
      1. For each pair, build a row dict with "model" plus the four metrics.
      2. Return pd.DataFrame(rows, columns=["model", "accuracy", "precision",
         "recall", "f1"]) so the column order is fixed and rows keep the
         order given.
    """
    rows = []
    for name, metrics in results:
        row = {"model": name}
        for key in ("accuracy", "precision", "recall", "f1"):
            row[key] = metrics[key]
        rows.append(row)
    return pd.DataFrame(rows, columns=["model", "accuracy", "precision", "recall", "f1"])

In [ ]:
comparison = attempt(
    "build_comparison_table",
    lambda: build_comparison_table([
        ("majority baseline", metrics_dummy),
        ("logistic regression", metrics_lr),
        ("LR class_weight=balanced", metrics_bal),
    ]),
)

if comparison is not None:
    print(comparison.to_string(index=False))

In [ ]:
_fx_results = [
    ("model_a", {"accuracy": 0.5, "precision": 0.4, "recall": 0.3, "f1": 0.2}),
    ("model_b", {"accuracy": 0.9, "precision": 0.8, "recall": 0.7, "f1": 0.6}),
]

check(
    "task 7: fixture table has the right shape and column order",
    lambda: build_comparison_table(_fx_results).shape == (2, 5)
    and list(build_comparison_table(_fx_results).columns)
    == ["model", "accuracy", "precision", "recall", "f1"],
)
check(
    "task 7: fixture rows keep their order and values",
    lambda: build_comparison_table(_fx_results).iloc[0].tolist() == ["model_a", 0.5, 0.4, 0.3, 0.2]
    and build_comparison_table(_fx_results).iloc[1].tolist() == ["model_b", 0.9, 0.8, 0.7, 0.6],
)
check(
    "task 7: real table matches the worked target output",
    lambda: comparison.iloc[1].tolist() == ["logistic regression", 0.92, 1.0, 0.2, 0.333]
    and comparison.iloc[2].tolist() == ["LR class_weight=balanced", 0.99, 0.909, 1.0, 0.952],
)
summary()

## Part 8: Wrap-up discussion and written summary

**Group discussion, tables of 3 to 4, ten minutes.** Argue from the table on your own screens, not from memory:

- The accuracy column reads 0.900, 0.920, 0.990. The recall column reads 0.000, 0.200, 1.000. A stakeholder saw only the accuracy column and concluded all three models are within a few points of each other. Walk through what that conclusion gets wrong about each row.
- The balanced model raises one false alarm per hundred tickets. A coordinator spends maybe ten minutes confirming a false alarm; a missed high-risk ticket is a customer with active water damage or a gas smell. Make the cost argument for shipping the balanced model, then steelman the strongest argument against it.
- The plain model's precision is a perfect 1.000, the best number anywhere in the table. Explain to a non-engineer why the model with the best precision in the table is the second-worst model in the table.

**Written summary, individually, 3 to 5 sentences in the cell below.** Cover: why a majority-class model can score high accuracy but zero F1; why F1 or precision plus recall beats accuracy for Cordwell's high-risk queue; and which model you would ship, with the trade you are accepting.

*Write your summary here.*

## Stretch goals (fast finishers)

Solutions live in the instructor solution notebook, released after the lab. Both hint files cover these at the same tiered depth as the main tasks.

**Stretch 1: The chance baseline.** `DummyClassifier(strategy="stratified", random_state=RANDOM_SEED)` guesses labels at random in proportion to the training base rates, a chance-level competitor to go with your floor-level one. Fit it, evaluate it with `evaluate_predictions`, and add it to the comparison table. Before you run it, predict: will its accuracy beat the majority baseline? Will its recall?

**Stretch 2: The ranking behind the labels.** Compute `predict_proba` scores on the test set for the plain Task 5 pipeline and feed them to `sklearn.metrics.average_precision_score`, then plot the curve from `precision_recall_curve`. Average precision summarizes ranking quality across all thresholds the way accuracy never can, and its floor for a random ranker is the positive rate, 0.10 here, not 0.5. The result for the plain model will surprise you given its recall; work out what it means before reading the solution.

**Stretch 3: The other lever.** This morning's threshold lesson, applied to imbalance: wrap a fresh Task 5 pipeline in `TunedThresholdClassifierCV(scoring="f1", cv=5)`, fit on the training data, read `best_threshold_`, and evaluate on the test set. You now have two distinct fixes for the same failure, reweighting the loss (Task 6) and moving the cutoff. Compare their test metrics and think about when an engineer would prefer each.

## Stretch goal solutions (instructor notebook only)

In [ ]:
# Stretch 1: chance-level baseline via strategy="stratified".
# This strategy DOES use random_state: it draws each prediction at random
# in proportion to the training base rates (about 90 percent routine,
# 10 percent high risk).
strat_clf = DummyClassifier(strategy="stratified", random_state=RANDOM_SEED)
strat_clf.fit(X_train, y_train)
y_pred_strat = strat_clf.predict(X_test)

metrics_strat, cm_strat = evaluate_predictions(y_test, y_pred_strat)
print("Stratified (chance-level) dummy on the test set")
print(cm_strat)
for k, v in metrics_strat.items():
    print(f"{k:<10} {v:.3f}")

full_table = build_comparison_table([
    ("majority baseline", metrics_dummy),
    ("stratified dummy", metrics_strat),
    ("logistic regression", metrics_lr),
    ("LR class_weight=balanced", metrics_bal),
])
print()
print(full_table.to_string(index=False))

# Reading: guessing at the base rate catches 1 of 10 real issues by luck,
# raises 9 false alarms, and its accuracy DROPS to 0.820, below the
# majority baseline. Random guessing pays an accuracy price for its
# occasional lucky catch. Note the recall ceiling of chance: about equal
# to the base rate, 0.1, which is the number any real model must clear.

In [ ]:
# Stretch 2: ranking quality via average precision and the PR curve.
from sklearn.metrics import precision_recall_curve, average_precision_score

proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]
ap_lr = average_precision_score(y_test, proba_lr)
print(f"Average precision, plain logistic regression: {ap_lr:.3f}")
print(f"Random-ranker floor (positive rate): {y_test.mean():.3f}")

print(f"Lowest score given to a real high-risk ticket : {proba_lr[y_test == 1].min():.4f}")
print(f"Highest score given to a routine ticket       : {proba_lr[y_test == 0].max():.4f}")

precision_curve, recall_curve, _ = precision_recall_curve(y_test, proba_lr)
fig, ax = plt.subplots(figsize=(5.2, 4.0))
ax.step(recall_curve, precision_curve, where="post")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title(f"Precision-recall curve, plain LR (AP = {ap_lr:.3f})")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Reading: average precision is 1.000. Every high-risk ticket received a
# higher score than every routine ticket; the model's RANKING of the test
# set is perfect. Its recall was 0.200 anyway because eight of the ten
# positives scored between 0.20 and 0.49, below the default 0.5 cutoff.
# The knowledge was there; the threshold threw it away. On real corpora
# the ranking will not be perfect, but the diagnosis pattern transfers:
# when average precision is high and recall is low, the fix is the
# decision rule, not more model. Which is Stretch 3.

In [ ]:
# Stretch 3: fix the cutoff instead of the loss.
from sklearn.model_selection import TunedThresholdClassifierCV

tuned_model = TunedThresholdClassifierCV(
    Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]),
    scoring="f1",
    cv=5,
)
tuned_model.fit(X_train, y_train)
print(f"Threshold chosen by cross validation on the training data: {tuned_model.best_threshold_:.3f}")

y_pred_tuned = tuned_model.predict(X_test)
metrics_tuned, cm_tuned = evaluate_predictions(y_test, y_pred_tuned)
print(cm_tuned)
for k, v in metrics_tuned.items():
    print(f"{k:<10} {v:.3f}")

# Reading: the CV search settled near 0.2, and at that cutoff this model
# scores perfectly on this test set, consistent with Stretch 2's perfect
# ranking. Do not generalize the perfection: it is a property of this
# synthetic corpus, where the two classes separate cleanly in score space.
# The durable lessons are (1) the threshold was tuned with cross
# validation inside the training data, so the test set still had no vote,
# which is how tuning is done when no separate validation set exists, and
# (2) you now hold two independent levers for imbalance, reweighting the
# loss and moving the cutoff. Reweighting changes what the model learns;
# thresholding changes how you act on what it learned. In production the
# threshold is usually the cheaper lever to revisit, because it can move
# without retraining, and sklearn ships it as FixedThresholdClassifier,
# which you met in this morning's stretch goals.

## Recap

- On imbalanced data, accuracy's starting line is the majority rate, not 50 percent. The dummy scored 0.900 here for free, so the plain model's 0.920 bought almost nothing, and the accuracy column compressed the entire story into two meaningless points.
- Precision and recall split "being right" into the two directions that carry different costs, and F1 refuses to be gamed by maxing one of them. The plain model's 1.000 precision next to 0.200 recall is this lab's proof that a single beautiful number can describe a nearly useless model.
- The majority baseline is the first row of every honest comparison table. Its accuracy is what the imbalance pays out for free; its zeros everywhere else are what intelligence has to earn.
- `class_weight="balanced"` changes what the training objective cares about, and one argument took recall from 0.200 to 1.000 at the price of one false alarm. Whether that price is worth paying is a business question, which is precisely why it cannot be delegated to the accuracy column.
- The same discipline as this morning still applies: one evaluation function for every model, the test set touched only to grade finished recipes, and every claim in the comparison traceable to a seeded, reproducible run.

In [ ]:
summary()
if CHECK_RESULTS and all(CHECK_RESULTS.values()):
    print("All checks passing. Lab complete.")